<a href="https://colab.research.google.com/github/jkshakthi999/Conditional-Image-Colorizer/blob/main/task5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

!pip install -q gradio scikit-image
print("Libraries installed.")

PyTorch: 2.14.1+cpu
GPU available: False
Libraries installed.


In [2]:
import os, io, re, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image, ImageOps
from skimage import color
from skimage.segmentation import flood
from scipy.ndimage import uniform_filter, gaussian_filter
import matplotlib.colors as mcolors
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/hint_colorization"      # photos + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/hint_colorization"

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32)
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
WORK_SIDE = 640       # size of the photo shown in the hint canvas
MIN_SAT = 0.03        # training photos with lower average saturation are treated as black & white and skipped
BATCH_SIZE = 16
LR = 3e-4
COLOR_W = 10.0        # weight of the colour loss
HINT_W = 5.0          # weight of the "follow the hints" loss
P_NO_HINT = 0.15      # share of training samples without any hint (keeps fully automatic colorization working)
P_REGION = 0.35       # share of training samples that also get "same-colour area" hints
HINT_STEP = 8         # spacing (px at model resolution) of the hint grid inside a selected area
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
PHOTO_DIR = os.path.join(DATA_ROOT, "photos")
CACHE_ROOT = "/content/hint_cache"                          # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "hint_colorizer.pt")         # trained model

os.makedirs(PHOTO_DIR, exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)

Mounted at /content/drive
Device: cpu
Data folder: /content/drive/MyDrive/hint_colorization


In [3]:
# ---------------- training photos ----------------
def prepare_photos():
    # reads the uploaded colour photos, skips bad / black & white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, "photos")
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(PHOTO_DIR, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white (training photos must be colour)"))
                continue
            k = 320.0 / min(w, h)
            if k < 1.0:
                im = im.resize((max(1, int(round(w * k))), max(1, int(round(h * k)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"photo_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


# ---------------- colour-hint utilities ----------------
PRESETS = {
    "Sky blue": "#87CEEB", "Deep blue": "#1F4E9C", "Grass green": "#5FAE3C", "Forest green": "#2E6B34",
    "Skin tone (light)": "#E3B79A", "Skin tone (dark)": "#8D5B3F", "Brown / wood": "#8B5A2B",
    "Sand / beige": "#D9C39A", "Red": "#D62F2F", "Yellow": "#F2D02F", "Orange": "#F28C28", "Pink": "#F08FB0",
    "Purple": "#7D4FB5", "Water teal": "#2A9DA8", "Stone gray": "#8A8A8A", "White / cloud": "#F2F2F2",
}


def parse_color(value):
    # colour name, hex code or rgb(...) text -> (r, g, b) integers, or None if it is not understood
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None
    m = re.match(r"rgba?\(\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)", s)
    if m:
        return tuple(int(round(min(255.0, float(m.group(i))))) for i in (1, 2, 3))
    for name, hx in PRESETS.items():
        if s == name.lower():
            s = hx.lower()
            break
    if re.fullmatch(r"[0-9a-f]{6}", s) or re.fullmatch(r"[0-9a-f]{3}", s):
        s = "#" + s
    try:
        r, g, b = mcolors.to_rgb(s.replace(" ", ""))
        return (int(round(r * 255)), int(round(g * 255)), int(round(b * 255)))
    except Exception:
        return None


def resolve_color(picker_value, text_value):
    # typed colour (if any) wins over the colour picker
    if text_value is not None and str(text_value).strip():
        c = parse_color(text_value)
        if c is None:
            raise gr.Error(f"I do not know the colour '{text_value}'. Use a name like 'sky blue' or a code like #87ceeb.")
        return c
    c = parse_color(picker_value)
    if c is None:
        raise gr.Error("Please pick a colour first.")
    return c


def rgb_to_ab(rgb):
    # the colour part (a, b) of an RGB colour in Lab space, scaled to ~[-1, 1] (the lightness comes from the photo)
    lab = color.rgb2lab(np.array([[rgb]], dtype=np.float64) / 255.0)[0, 0]
    return (lab[1:] / 128.0).astype(np.float32)


def sample_hints(ab, rng, YY, XX):
    # simulates user hints from a real colour image. ab: H x W x 2 in [-1, 1]  ->  H x W x 3 = [hint_a, hint_b, mask]
    H, W = ab.shape[:2]
    hint = np.zeros((H, W, 3), np.float32)
    if rng.random() < P_NO_HINT:
        return hint
    for _ in range(min(int(rng.geometric(1.0 / 8.0)), 40)):                    # random colour spots
        cy, cx = int(rng.integers(0, H)), int(rng.integers(0, W))
        rad = int(rng.integers(1, 9))
        y0, y1, x0, x1 = max(0, cy - rad), min(H, cy + rad + 1), max(0, cx - rad), min(W, cx + rad + 1)
        sub = ((YY[y0:y1, x0:x1] - cy) ** 2 + (XX[y0:y1, x0:x1] - cx) ** 2) <= rad * rad
        col = ab[y0:y1, x0:x1][sub].mean(0) + rng.normal(0, 0.03, 2)
        view = hint[y0:y1, x0:x1]
        view[sub, 0:2] = col
        view[sub, 2] = 1.0
    if rng.random() < P_REGION:                                                # "this whole area has this colour"
        for _ in range(int(rng.integers(1, 4))):
            cy, cx = int(rng.integers(0, H)), int(rng.integers(0, W))
            R = int(rng.integers(20, 71))
            y0, y1, x0, x1 = max(0, cy - R), min(H, cy + R), max(0, cx - R), min(W, cx + R)
            c0 = ab[cy, cx]
            close = np.linalg.norm(ab[y0:y1, x0:x1] - c0, axis=-1) < rng.uniform(0.04, 0.10)
            step = int(rng.choice([6, 8, 10, 12]))
            col = c0 + rng.normal(0, 0.03, 2)
            for py in range(y0 + step // 2, y1, step):
                for px in range(x0 + step // 2, x1, step):
                    if close[py - y0, px - x0]:
                        ya, xa = max(0, py - 2), max(0, px - 2)
                        hint[ya:py + 3, xa:px + 3, 0:2] = col
                        hint[ya:py + 3, xa:px + 3, 2] = 1.0
    return hint


def build_hint_array(hints, h, w):
    # the user's hints -> H x W x 3 array [hint_a, hint_b, mask] at the model resolution
    hint = np.zeros((h, w, 3), np.float32)
    yy, xx = np.mgrid[0:h, 0:w]
    for k in hints:
        ab = rgb_to_ab(k["rgb"])
        if k["kind"] == "point":
            cx, cy = k["x"] * (w - 1), k["y"] * (h - 1)
            rad = max(1.0, float(k["size"]))
            disc = (yy - cy) ** 2 + (xx - cx) ** 2 <= rad * rad
            hint[disc, 0:2] = ab
            hint[disc, 2] = 1.0
        else:
            m = np.asarray(Image.fromarray(k["mask"].astype(np.uint8) * 255).resize((w, h), Image.NEAREST)) > 127
            grid = np.zeros((h, w), bool)
            grid[HINT_STEP // 2::HINT_STEP, HINT_STEP // 2::HINT_STEP] = True
            pts = np.argwhere(m & grid)
            if len(pts) == 0 and m.any():                                       # tiny area: use a few of its pixels
                pts = np.argwhere(m)[::max(1, int(m.sum()) // 3)][:3]
            for py, px in pts:
                ya, xa = max(0, py - 2), max(0, px - 2)
                hint[ya:py + 3, xa:px + 3, 0:2] = ab
                hint[ya:py + 3, xa:px + 3, 2] = 1.0
    return hint


def enforce_hints(ab, hints, enforce):
    # blends the requested colours into the chosen areas (ab: H x W x 2 in ~[-1, 1])
    if enforce <= 0 or not hints:
        return ab
    H, W = ab.shape[:2]
    ab = np.ascontiguousarray(ab, dtype=np.float32).copy()
    for k in hints:
        c = rgb_to_ab(k["rgb"])
        if k["kind"] == "point":
            cx, cy = k["x"] * (W - 1), k["y"] * (H - 1)
            rad = max(2.0, k["size"] / float(INFER_SIDE) * max(H, W))
            pad = int(3 * rad) + 2
            y0, y1 = max(0, int(cy) - pad), min(H, int(cy) + pad + 1)
            x0, x1 = max(0, int(cx) - pad), min(W, int(cx) + pad + 1)
            yy, xx = np.mgrid[y0:y1, x0:x1]
            m = ((yy - cy) ** 2 + (xx - cx) ** 2 <= rad * rad).astype(np.float32)
            m = gaussian_filter(m, max(1.0, 0.35 * rad))[..., None] * enforce
            sub = ab[y0:y1, x0:x1]
            sub[:] = sub * (1.0 - m) + c * m
        else:
            m = np.asarray(Image.fromarray(k["mask"].astype(np.uint8) * 255).resize((W, H), Image.BILINEAR)).astype(np.float32) / 255.0
            m = gaussian_filter(m, max(1.0, 0.004 * max(H, W)))[..., None] * enforce
            ab = ab * (1.0 - m) + c * m
    return ab


def guided_filter(I, p, r, eps):
    # edge-aware smoothing of p guided by I (He et al.): colours follow the brightness edges of the photo
    size = 2 * r + 1
    mean_I, mean_p = uniform_filter(I, size), uniform_filter(p, size)
    var_I = uniform_filter(I * I, size) - mean_I * mean_I
    cov_Ip = uniform_filter(I * p, size) - mean_I * mean_p
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return uniform_filter(a, size) * I + uniform_filter(b, size)


# ---------------- dataset ----------------
class HintDataset(Dataset):
    def __init__(self, paths, train=True):
        self.paths = paths
        self.train = train
        self.YY, self.XX = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        ab = lab[..., 1:] / 128.0
        rng = np.random.default_rng(None if self.train else 1000 + i)           # fixed hints for validation
        hint = sample_hints(ab, rng, self.YY, self.XX)
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)        # lightness -> [-1, 1]
        return L, torch.from_numpy(ab).permute(2, 0, 1), torch.from_numpy(hint).permute(2, 0, 1)


# ---------------- model ----------------
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch + 3, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))

    def forward(self, x, skip, hint):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        h = F.interpolate(hint, size=skip.shape[-2:], mode="area")              # hints are fed to every decoder stage
        return self.conv(torch.cat([x, skip, h], dim=1))


class HintColorizer(nn.Module):
    # grayscale (L) + colour hints (hint_a, hint_b, mask)  ->  colour (ab)
    def __init__(self, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        conv1 = nn.Conv2d(4, 64, 7, 2, 3, bias=False)                           # input: L + 3 hint channels
        with torch.no_grad():
            conv1.weight.zero_()
            conv1.weight[:, 0:1] = r.conv1.weight.sum(dim=1, keepdim=True)      # L channel = pretrained RGB weights summed
        self.stem = nn.Sequential(conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.up3 = UpBlock(512, 256, 256)
        self.up2 = UpBlock(256, 128, 128)
        self.up1 = UpBlock(128, 64, 64)
        self.up0 = UpBlock(64, 64, 64)
        self.head = nn.Sequential(nn.Conv2d(64 + 1 + 3, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())

    def forward(self, L, hint):
        s0 = self.stem(torch.cat([L, hint], dim=1))
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.l4(s3)
        d = self.up3(s4, s3, hint)
        d = self.up2(d, s2, hint)
        d = self.up1(d, s1, hint)
        d = self.up0(d, s0, hint)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, L, hint], dim=1))                        # ab in [-1, 1]

print("Photo, hint and model code ready.")

Photo, hint and model code ready.


In [4]:
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)


def count_photos():
    if not os.path.isdir(PHOTO_DIR):
        return 0
    return len([f for f in os.listdir(PHOTO_DIR) if f.lower().endswith(EXTS)])


def dataset_status():
    n = count_photos()
    note = "" if n >= 100 else ("  (more is better: 100+ recommended, 20 minimum)" if n > 0 else "  (none yet)")
    return f"**Training photos saved in Drive:** {n}{note}"


def save_uploads(uploaded):
    if not uploaded:
        return "No photos selected. Click the upload box, choose photos, then click Save.", dataset_status(), None
    os.makedirs(PHOTO_DIR, exist_ok=True)
    ok = bad = 0
    stamp = int(time.time())
    for i, f in enumerate(uploaded):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(PHOTO_DIR, f"up_{stamp}_{i:04d}{os.path.splitext(path)[1].lower()}"))
        ok += 1
    msg = f"Saved **{ok}** photo(s)." + (f" Skipped {bad} (not valid images)." if bad else "")
    return msg + " Upload more, or go to the Train Model tab.", dataset_status(), None


def masked_l1(pred, hint):
    # how far the prediction is from the hinted colours, only at hinted pixels
    m = hint[:, 2:3]
    return ((pred - hint[:, 0:2]).abs() * m).sum() / (m.sum() * 2.0 + 1e-6)


def train_model(epochs):
    global model
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    paths, bad = prepare_photos()
    yield emit(f"  usable colour photos: {len(paths)}   skipped: {len(bad)}")
    for name, why in bad[:3]:
        yield emit(f"      skipped {name}: {why}")
    if len(paths) < 8:
        raise gr.Error("Fewer than 8 usable colour photos. Upload more colour photos in the first tab.")
    if len(paths) < 20:
        yield emit("    WARNING: fewer than 20 photos; results will be weak. 100+ is recommended.")

    random.shuffle(paths)
    n_val = max(2, int(0.1 * len(paths)))
    train_ds = HintDataset(paths[n_val:], train=True)
    val_ds = HintDataset(paths[:n_val], train=False)
    bs = min(BATCH_SIZE, len(train_ds))
    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training photos: {len(train_ds)} | validation photos: {len(val_ds)} | batch size: {bs}")

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = HintColorizer(pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    color_loss = nn.SmoothL1Loss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for L, ab, hint in train_dl:
            L, ab, hint = L.to(device), ab.to(device), hint.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred = net(L, hint)
            pred = pred.float()
            loss = COLOR_W * color_loss(pred, ab) + HINT_W * masked_l1(pred, hint)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        v_h = v_0 = 0.0
        with torch.no_grad():
            for L, ab, hint in val_dl:
                L, ab, hint = L.to(device), ab.to(device), hint.to(device)
                v_h += color_loss(net(L, hint).float(), ab).item()
                v_0 += color_loss(net(L, torch.zeros_like(hint)).float(), ab).item()
        v_h /= len(val_dl)
        v_0 /= len(val_dl)
        va = COLOR_W * (v_h + v_0) / 2.0

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict()}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.3f}   val error with hints {v_h:.4f}   "
                   f"without hints {v_0:.4f}   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    e_h = e_0 = 0.0
    n = 0
    with torch.no_grad():
        for L, ab, hint in val_dl:
            L, ab, hint = L.to(device), ab.to(device), hint.to(device)
            e_h += (net(L, hint).float() - ab).abs().mean().item() * len(L)
            e_0 += (net(L, torch.zeros_like(hint)).float() - ab).abs().mean().item() * len(L)
            n += len(L)
    e_h /= max(1, n)
    e_0 /= max(1, n)
    yield emit(f"\nAverage colour error on validation photos: without hints {e_0:.4f}  ->  with simulated hints {e_h:.4f} "
               f"({100.0 * (e_0 - e_h) / max(e_0, 1e-9):.0f}% lower). Lower is better.")

    model = net
    yield emit("\nDONE. Model saved to Drive. Go to the 'Colorize with your colour hints' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [5]:
def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def _prep(img):
    # PIL image -> model input tensor + full-resolution lightness
    img = ImageOps.exif_transpose(img).convert("RGB")
    W, H = img.size
    if max(W, H) > MAX_SIDE:
        k = MAX_SIDE / float(max(W, H))
        W, H = max(1, int(W * k)), max(1, int(H * k))
        img = img.resize((W, H), Image.LANCZOS)
    L_full = color.rgb2lab(np.asarray(img))[..., 0].astype(np.float32)        # 0..100 at full resolution
    k = INFER_SIDE / float(max(W, H))
    small = img.resize((_round32(W * k), _round32(H * k)), Image.BICUBIC)
    L = color.rgb2lab(np.asarray(small))[..., 0] / 50.0 - 1.0
    x = torch.from_numpy(L).float()[None, None].to(device)
    return x, L_full, H, W


def _check_ready():
    if model is None:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")


@torch.no_grad()
def colorize_with_hints(img, hints, saturation=1.2, enforce=0.7, refine=0.7):
    # hints: list of the user's colour hints (empty list = fully automatic colorization)
    # enforce: 0..1 how strongly the requested colours are forced into the chosen areas
    # refine:  0..1 edge-aware colour refinement
    _check_ready()
    model.eval()
    x, L_full, H, W = _prep(img)
    h, w = x.shape[-2:]
    hint = torch.from_numpy(build_hint_array(hints, h, w)).permute(2, 0, 1)[None].to(device)
    ab = model(x, hint).float().cpu()                                           # 1 x 2 x h x w in [-1, 1]
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)[0].permute(1, 2, 0).numpy()
    ab = np.ascontiguousarray(ab, dtype=np.float32) * saturation               # colour strength
    ab = enforce_hints(ab, hints, enforce)                                      # the requested colours in the chosen areas
    if refine > 0:                                                              # snap colours to the photo's brightness edges
        guide = (L_full / 100.0).astype(np.float32)
        r = max(2, int(round(max(H, W) / 150.0)))
        smooth = np.stack([guided_filter(guide, np.ascontiguousarray(ab[..., k]), r, 1e-3) for k in range(2)], axis=-1)
        ab = (1.0 - refine) * ab + refine * smooth
    ab = (ab * 128.0).clip(-128, 127)
    lab = np.concatenate([L_full[..., None], ab], axis=-1)
    rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        model = HintColorizer(pretrained=False).to(device)
        model.load_state_dict(ck["model"])
        model.eval()
        print("Loaded saved model from Drive.")
    except Exception as e:
        model = None
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload photos and train inside the app (next cell).")

No saved model yet. Upload photos and train inside the app (next cell).


In [ ]:
try:
    demo.close()          # close an older app if this cell is re-run
except Exception:
    pass

MODE_DOT, MODE_WAND, MODE_RECT = "dot", "wand", "rect"


# ---------------- hint canvas helpers ----------------
def hints_html(state):
    hs = state.get("hints", []) if state else []
    if not hs:
        return "<i>No colour hints yet. Without hints the model colorizes automatically.</i>"
    rows = []
    for i, h in enumerate(hs, 1):
        hexc = "#%02x%02x%02x" % tuple(h["rgb"])
        desc = (f"Dot at ({h['x'] * 100:.0f}%, {h['y'] * 100:.0f}%), size {h['size']}" if h["kind"] == "point" else h["label"])
        rows.append(f"<div>{i}. <span style='display:inline-block;width:14px;height:14px;background:{hexc};"
                    f"border:1px solid #888;vertical-align:middle'></span> {hexc} - {desc}</div>")
    if state.get("pending"):
        rows.append("<div><b>Rectangle: first corner set. Click the opposite corner.</b></div>")
    return "".join(rows)


def render_overlay(state):
    # the grayscale photo with the user's hints drawn on it
    arr = np.asarray(state["work"].convert("RGB")).astype(np.float32).copy()
    Hd, Wd = arr.shape[:2]
    yy, xx = np.mgrid[0:Hd, 0:Wd]
    for h in state["hints"]:
        c = np.array(h["rgb"], np.float32)
        if h["kind"] == "region":
            m = h["mask"]
            arr[m] = 0.5 * arr[m] + 0.5 * c
        else:
            cx, cy = h["x"] * (Wd - 1), h["y"] * (Hd - 1)
            rad = max(3.0, h["size"] / float(INFER_SIDE) * max(Hd, Wd))
            d2 = (yy - cy) ** 2 + (xx - cx) ** 2
            arr[(d2 <= (rad + 2) ** 2) & (d2 > rad * rad)] = 255.0
            arr[d2 <= rad * rad] = c
    if state.get("pending"):
        px, py = int(state["pending"][0] * (Wd - 1)), int(state["pending"][1] * (Hd - 1))
        arr[max(0, py - 6):py + 7, px] = (255, 0, 255)
        arr[py, max(0, px - 6):px + 7] = (255, 0, 255)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def add_hint(state, nx, ny, mode, size, tol, rgb):
    Wd, Hd = state["work"].size
    nx, ny = float(min(max(nx, 0.0), 1.0)), float(min(max(ny, 0.0), 1.0))
    rgb = tuple(int(v) for v in rgb)
    hexc = "#%02x%02x%02x" % rgb
    row, col = int(round(ny * (Hd - 1))), int(round(nx * (Wd - 1)))
    if mode == MODE_DOT:
        state["hints"].append({"kind": "point", "x": nx, "y": ny, "size": int(size), "rgb": rgb})
        return f"Added a {hexc} dot. Add more hints, or click Colorize."
    if mode == MODE_WAND:
        m = flood(state["gray"], (row, col), tolerance=float(tol) / 100.0)
        frac = float(m.mean())
        state["hints"].append({"kind": "region", "mask": m, "rgb": rgb,
                               "label": f"Magic-wand area ({frac * 100:.0f}% of the photo)"})
        msg = f"Selected an area covering {frac * 100:.0f}% of the photo and coloured it {hexc}."
        if frac > 0.6:
            msg += " That is very large; lower the tolerance (or undo) if it covers too much."
        elif frac < 0.002:
            msg += " That is tiny; raise the tolerance (or undo) to select more."
        return msg
    if state.get("pending") is None:                                            # rectangle: first corner
        state["pending"] = (nx, ny)
        return "First corner set (pink cross). Now click the opposite corner."
    x0, y0 = state["pending"]
    state["pending"] = None
    c0, c1 = sorted([int(round(x0 * (Wd - 1))), col])
    r0, r1 = sorted([int(round(y0 * (Hd - 1))), row])
    m = np.zeros((Hd, Wd), bool)
    m[r0:r1 + 1, c0:c1 + 1] = True
    state["hints"].append({"kind": "region", "mask": m, "rgb": rgb,
                           "label": f"Rectangle ({float(m.mean()) * 100:.0f}% of the photo)"})
    return f"Rectangle added with colour {hexc}."


# ---------------- event handlers ----------------
def on_image_change(img, state):
    if img is None:
        return None, None, hints_html(None), "Upload a photo to begin."
    base = ImageOps.exif_transpose(img).convert("RGB")
    work = base.copy()
    work.thumbnail((WORK_SIDE, WORK_SIDE), Image.LANCZOS)
    gray = work.convert("L")
    st = {"base": base, "work": Image.merge("RGB", (gray, gray, gray)),
          "gray": gaussian_filter(np.asarray(gray).astype(np.float32) / 255.0, 1.5), "hints": [], "pending": None}
    return render_overlay(st), st, hints_html(st), "Photo loaded. Choose a colour and a tool, then click on the photo on the right."


def on_click(state, mode, size, tol, cpick, ctext, evt: gr.SelectData):
    if not state or "work" not in state:
        raise gr.Error("Upload a photo first.")
    Wd, Hd = state["work"].size
    rgb = resolve_color(cpick, ctext)
    msg = add_hint(state, evt.index[0] / float(max(1, Wd - 1)), evt.index[1] / float(max(1, Hd - 1)), mode, size, tol, rgb)
    return render_overlay(state), state, hints_html(state), msg


def on_add_xy(state, xpct, ypct, mode, size, tol, cpick, ctext):
    if not state or "work" not in state:
        raise gr.Error("Upload a photo first.")
    rgb = resolve_color(cpick, ctext)
    msg = add_hint(state, xpct / 100.0, ypct / 100.0, mode, size, tol, rgb)
    return render_overlay(state), state, hints_html(state), msg


def on_undo(state):
    if not state or "work" not in state:
        return None, state, hints_html(None), "Upload a photo first."
    if state["hints"]:
        state["hints"].pop()
    state["pending"] = None
    return render_overlay(state), state, hints_html(state), "Removed the last hint."


def on_clear(state):
    if not state or "work" not in state:
        return None, state, hints_html(None), "Upload a photo first."
    state["hints"] = []
    state["pending"] = None
    return render_overlay(state), state, hints_html(state), "All hints cleared."


def run_colorize(state, saturation, enforce, refine):
    if not state or "base" not in state:
        raise gr.Error("Please upload a photo first.")
    try:
        t0 = time.time()
        base, hints = state["base"], state["hints"]
        auto = colorize_with_hints(base, [], saturation, enforce, refine)
        res = colorize_with_hints(base, hints, saturation, enforce, refine) if hints else auto
        orig = base if base.size == res.size else base.resize(res.size, Image.LANCZOS)
        note = (f"Colorized using **{len(hints)}** colour hint(s) in {time.time() - t0:.1f} seconds." if hints
                else "No hints were added, so this is the automatic colorization. Click on the photo to add colour hints.")
        return res, orig, res, auto, res, note
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


def model_status():
    global model
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            m = HintColorizer(pretrained=False).to(device)
            m.load_state_dict(ck["model"])
            m.eval()
            model = m
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None:
        return "⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** (upload colour photos in tab 1 first)."
    return "✅ **Model ready.**"


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🎨 Conditional Image Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Tell the model what colour each area should be</b> &bull; "
    "<b>Pick or type colours</b> &bull; <b>Click to mark areas</b> &bull; <b>Compare with and without hints</b></p>"
)

with gr.Blocks(title="Conditional Image Colorizer") as demo:
    gr.HTML(HEADER)
    state = gr.State(None)
    with gr.Tabs():

        # ---------------- Tab 1: upload training photos ----------------
        with gr.Tab("1. Upload Training Photos"):
            gr.Markdown("Upload **real colour photos** of any scenes (sky, grass, people, buildings, water ...). The model learns natural colours "
                        "and how to follow colour hints. Use varied photos. Aim for 100+ (20 minimum).")
            up_files = gr.File(label="Choose colour photos", file_count="multiple", file_types=["image"])
            up_btn = gr.Button("Save photos", variant="primary")
            up_msg = gr.Markdown()
            ds_md = gr.Markdown(dataset_status())
            up_btn.click(save_uploads, inputs=[up_files], outputs=[up_msg, ds_md, up_files])

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading photos. It takes roughly 5-20 minutes on a T4 GPU. "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize with hints ----------------
        with gr.Tab("3. Colorize with your colour hints"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="📷 1) Upload a B&W photo", height=260)
                    gr.Markdown("**2) Choose the colour for an area**")
                    preset_dd = gr.Dropdown(choices=list(PRESETS), value="Sky blue", label="Preset colour")
                    cpick = gr.ColorPicker(value=PRESETS["Sky blue"], label="Or pick any colour")
                    ctext = gr.Textbox(label="Or type a colour name / hex code (overrides the picker)",
                                       placeholder="e.g. grass green, skyblue, #87ceeb")
                    gr.Markdown("**3) Choose a tool, then click on the photo (right) to mark the area**")
                    mode_rd = gr.Radio(choices=[("● Dot (small spot)", MODE_DOT),
                                                ("🪄 Magic wand (select the area with similar brightness)", MODE_WAND),
                                                ("▭ Rectangle (click two opposite corners)", MODE_RECT)],
                                       value=MODE_WAND, label="Hint tool")
                    size_sl = gr.Slider(1, 12, value=4, step=1, label="Dot size")
                    tol_sl = gr.Slider(2, 40, value=12, step=1, label="Magic wand tolerance (%): higher = bigger area")
                    with gr.Row():
                        undo_btn = gr.Button("Undo last hint")
                        clear_btn = gr.Button("Clear all hints")
                    gr.Markdown("**4) Colorize**")
                    enforce_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Hint strength (how strongly your colours are applied)")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Overall colour strength")
                    refine_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Edge-aware colour refinement")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    with gr.Accordion("Can't click on the photo? Add a hint by position", open=False):
                        x_sl = gr.Slider(0, 100, value=50, step=1, label="X position (% from the left)")
                        y_sl = gr.Slider(0, 100, value=20, step=1, label="Y position (% from the top)")
                        add_btn = gr.Button("Add hint at this position")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload a photo, then click on it to add colour hints.")
                    canvas = gr.Image(type="pil", interactive=False, height=420, label="👆 Click on the photo to add colour hints")
                    hint_list = gr.HTML(hints_html(None))
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("With vs Without Hints"):
                            gr.Markdown("<p style='text-align:center'><b>Automatic (no hints) (left) | With your colour hints (right)</b></p>")
                            with gr.Row():
                                out_auto = gr.Image(type="pil", label="Automatic (no hints)", height=480)
                                out_hint = gr.Image(type="pil", label="With your hints", height=480)

    # ---------------- events ----------------
    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(model_status, None, model_md)
    demo.load(model_status, None, model_md)
    inp.change(on_image_change, [inp, state], [canvas, state, hint_list, status])
    preset_dd.change(lambda name: PRESETS[name], preset_dd, cpick)
    canvas.select(on_click, [state, mode_rd, size_sl, tol_sl, cpick, ctext], [canvas, state, hint_list, status])
    add_btn.click(on_add_xy, [state, x_sl, y_sl, mode_rd, size_sl, tol_sl, cpick, ctext], [canvas, state, hint_list, status])
    undo_btn.click(on_undo, [state], [canvas, state, hint_list, status])
    clear_btn.click(on_clear, [state], [canvas, state, hint_list, status])
    btn.click(run_colorize, [state, sat, enforce_sl, refine_sl], [out_main, out_orig, out_col, out_auto, out_hint, status])

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ceee7e6a5beacb22b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 112MB/s]
